In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import io
import os

In [ ]:
uploader = widgets.FileUpload(accept='image/*', multiple=False)
out = widgets.Output()

image_grey = None  # stores the PIL grayscale image globally

def on_upload_change(change):
    global image_grey
    with out:
        clear_output()
        if not change['new']:
            return
        file_info = change['new'][0]
        input_image = Image.open(io.BytesIO(file_info['content']))
        image_grey = input_image.convert('L')
        print(f"Loaded: {file_info['name']}")
        print(f"Size  : {image_grey.size}  (width × height)")
        display(image_grey)

uploader.observe(on_upload_change, names='value')
display(uploader, out)

In [ ]:
# Convert the uploaded grayscale PIL image to a NumPy float array.
# We use float64 so convolution arithmetic does not overflow.
gray_array = np.array(image_grey, dtype=np.float64)

print(f"Image shape : {gray_array.shape}  (rows × columns)")
print(f"Pixel range : [{gray_array.min():.0f}, {gray_array.max():.0f}]")
print(f"\nTop-left 5×5 pixel values:")
print(gray_array[:5, :5])

In [ ]:
def convolve2d(image, kernel):
    """
    2-D discrete convolution with zero-padding.
    Output has the same shape as the input image.
    """
    kH, kW = kernel.shape
    pH, pW = kH // 2, kW // 2

    # Zero-pad so border pixels are handled correctly
    padded = np.pad(image, ((pH, pH), (pW, pW)), mode='constant', constant_values=0)

    H, W   = image.shape
    output = np.zeros_like(image, dtype=np.float64)

    for i in range(H):
        for j in range(W):
            patch        = padded[i : i + kH, j : j + kW]
            output[i, j] = np.sum(patch * kernel)   # element-wise multiply then sum

    return output


def clip_and_convert(arr):
    """Clip values to valid pixel range [0, 255] and cast to uint8."""
    return np.clip(arr, 0, 255).astype(np.uint8)


print("convolve2d and clip_and_convert are ready.")

In [ ]:
# Emboss — 3-D relief effect
emboss_kernel = np.array([
    [-2, -1,  0],
    [-1,  1,  1],
    [ 0,  1,  2]], dtype=np.float64)

# Laplacian — outline / edge detection
laplacian_kernel = np.array([
    [-1, -1, -1],
    [-1,  8, -1],
    [-1, -1, -1]], dtype=np.float64)

print("Emboss kernel:")
print(emboss_kernel)
print("\nLaplacian kernel:")
print(laplacian_kernel)

In [ ]:
emboss_result    = convolve2d(gray_array, emboss_kernel)
laplacian_result = convolve2d(gray_array, laplacian_kernel)

print(f"Emboss    output range : [{emboss_result.min():.1f},  {emboss_result.max():.1f}]")
print(f"Laplacian output range : [{laplacian_result.min():.1f}, {laplacian_result.max():.1f}]")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].imshow(clip_and_convert(gray_array),       cmap='gray')
axes[0].set_title('Original Grayscale')

axes[1].imshow(clip_and_convert(emboss_result),    cmap='gray')
axes[1].set_title('Emboss (3-D Relief Effect)')

axes[2].imshow(clip_and_convert(laplacian_result), cmap='gray')
axes[2].set_title('Edge Detection (Laplacian)')

for ax in axes:
    ax.axis('off')

plt.suptitle('Emboss vs. Edge Detection', fontweight='bold')
plt.tight_layout()
plt.savefig('emboss_and_edge_detection.png', dpi=150)
plt.show()
print('Figure saved: emboss_and_edge_detection.png')